# 01 — Raw Data Exploration
Loads every raw series from the parquet cache, verifies date ranges, and plots each series.
Run `python -m src.data.fetch` first to populate `data/raw/`.

In [ ]:
import sys
from pathlib import Path

# Ensure project root is on the path when running from notebooks/
project_root = Path().resolve().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import pandas as pd

from src.config import ALL_SERIES, GDP_SERIES, PREDICTOR_SERIES, SAMPLE_START
from src.data.fetch import load_raw

In [ ]:
# ---------------------------------------------------------------------------
# Load all series from cache
# ---------------------------------------------------------------------------
data = {}
missing = []
for sid in ALL_SERIES:
    try:
        data[sid] = load_raw(sid)
    except FileNotFoundError:
        missing.append(sid)

if missing:
    print(f"NOT CACHED (run `python -m src.data.fetch`): {missing}")
else:
    print(f"All {len(data)} series loaded from cache.")

In [ ]:
# ---------------------------------------------------------------------------
# Date-range summary table
# ---------------------------------------------------------------------------
rows = []
for sid, s in data.items():
    freq = "Q" if sid == GDP_SERIES else "M"
    desc = PREDICTOR_SERIES.get(sid, {}).get("desc", "Real GDP (target)")
    short_history = str(s.index[0]) > "1990-01" if freq == "M" else str(s.index[0]) > "1990Q1"
    rows.append({
        "Series": sid,
        "Description": desc,
        "Freq": freq,
        "Start": str(s.index[0]),
        "End": str(s.index[-1]),
        "N": len(s),
        "Short history?": "YES" if short_history else "",
    })

summary = pd.DataFrame(rows).set_index("Series")
print(summary.to_string())

short = summary[summary["Short history?"] == "YES"]
if not short.empty:
    print("\nSeries with histories starting after 1990-01:")
    print(short[["Description", "Start"]].to_string())

In [ ]:
# ---------------------------------------------------------------------------
# Plot: GDP target
# ---------------------------------------------------------------------------
gdp = data[GDP_SERIES]
gdp_dt = gdp.copy()
gdp_dt.index = gdp_dt.index.to_timestamp()

fig, ax = plt.subplots(figsize=(12, 3))
ax.plot(gdp_dt.index, gdp_dt.values, color="steelblue", linewidth=1.5)
ax.set_title("GDPC1 — Real GDP (Billions of Chained 2017 Dollars, SA)")
ax.set_ylabel("Billions USD")
ax.xaxis.set_major_locator(mdates.YearLocator(5))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# ---------------------------------------------------------------------------
# Plot: all 12 monthly predictors in a grid
# ---------------------------------------------------------------------------
predictor_ids = list(PREDICTOR_SERIES.keys())
ncols = 3
nrows = (len(predictor_ids) + ncols - 1) // ncols

fig, axes = plt.subplots(nrows, ncols, figsize=(16, nrows * 3))
axes_flat = axes.flatten()

for i, sid in enumerate(predictor_ids):
    ax = axes_flat[i]
    s = data[sid].copy()
    s.index = s.index.to_timestamp()
    ax.plot(s.index, s.values, linewidth=1, color="darkorange")
    desc = PREDICTOR_SERIES[sid]["desc"]
    ax.set_title(f"{sid} — {desc}", fontsize=9)
    ax.xaxis.set_major_locator(mdates.YearLocator(10))
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    ax.tick_params(axis="x", labelsize=8)
    ax.grid(axis="y", alpha=0.3)

# Hide any unused subplots
for j in range(len(predictor_ids), len(axes_flat)):
    axes_flat[j].set_visible(False)

fig.suptitle("Monthly Predictor Series (raw levels)", fontsize=12, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# ---------------------------------------------------------------------------
# Missingness check: count NaNs per series
# ---------------------------------------------------------------------------
print("NaN counts per series:")
for sid, s in data.items():
    n_nan = s.isna().sum()
    flag = "  <-- HAS NANS" if n_nan > 0 else ""
    print(f"  {sid:<12} {n_nan:>4} NaNs{flag}")